# Manhwa Studio — Renderizador CPU no Google Colab
Este notebook inicia uma API temporária de renderização com FFmpeg. Conecte o ambiente e execute as células em ordem. Ao final, copie a URL HTTPS e o token para os campos de conexão do site. Mantenha esta aba aberta durante a renderização.

Use somente vídeos e imagens que você tem direito de processar. A sessão e os arquivos temporários são apagados quando o Colab encerra.

In [ ]:
# 1) Instalar FFmpeg, Cloudflared e dependências da API
!apt-get -qq update
!apt-get -qq install -y ffmpeg curl
!pip -q install fastapi uvicorn python-multipart

In [ ]:
# 2) Baixar o Cloudflare Tunnel para criar uma URL HTTPS temporária
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# 3) Baixar a API revisada do próprio repositório
!curl -fsSL https://raw.githubusercontent.com/manhwavaz1-lab/manhwa-recreator-studio/main/colab/manhwa_cpu_api.py -o /content/manhwa_cpu_api.py
!python -m py_compile /content/manhwa_cpu_api.py
print('API validada e pronta para iniciar na próxima célula.')


In [ ]:
# 3) Iniciar API + túnel público HTTPS. Execute esta célula por último.
import os, secrets, subprocess, time, re, threading
from IPython.display import display, Markdown
TOKEN = secrets.token_urlsafe(32)
os.environ['MANHWA_RENDER_TOKEN'] = TOKEN
api = subprocess.Popen(['uvicorn','manhwa_cpu_api:app','--host','0.0.0.0','--port','8000'],cwd='/content',stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000','--no-autoupdate'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
public_url = None
deadline=time.time()+90
while time.time()<deadline and tunnel.poll() is None:
    line=tunnel.stdout.readline()
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',line)
    if match: public_url=match.group(0); break
    time.sleep(.2)
if not public_url: raise RuntimeError('Não foi possível criar o túnel. Interrompa e execute novamente esta célula.')
display(Markdown('## Renderizador CPU conectado\n**URL temporária:** `'+public_url+'`\n\n**Token de sessão:** `'+TOKEN+'`\n\n**Health:** `'+public_url+'/health`\n\nMantenha esta célula em execução. A URL e o token mudam quando uma nova sessão é iniciada.'))
print('Servidor iniciado. Não feche esta aba durante o processamento.')

## Como usar
1. Abra o notebook e clique em **Conectar**.
2. Execute todas as células de cima para baixo.
3. Na célula de inicialização, copie a URL HTTPS e o token exibidos.
4. No site, abra **Conexão Google Colab**, cole os dois valores e clique em **Testar conexão**.
5. Com o indicador conectado, selecione o vídeo e a moldura e clique em **Criar vídeo**.

A API usa FFmpeg com `libx264` em CPU. O Colab gratuito pode desconectar, limitar recursos ou encerrar sessões longas. A URL/token mudam em cada sessão; não os compartilhe.

In [ ]:
# Opcional: parar os processos ao terminar
for p in (api, tunnel):
    try: p.terminate()
    except Exception: pass
print('Servidor e túnel encerrados.')